In [1]:
import sys
sys.path.append('../')

import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
from tqdm import tqdm
from matplotlib.colors import LogNorm
import pytz, cmath, itertools
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import scipy.sparse as ssp
from sympy import symbols
import utils_2Q_gate_zp as ut
import pandas as pd
import scipy as sp
from multiprocessing import Pool
import qutip as qt
import multiprocessing as mp
from multiprocessing import Pool
import scqubits as scq
from sympy import symbols
import scipy.sparse as ssp
from datetime import datetime

# Coupled zero pi

In [2]:
truc_full = 500
num_cpus, n_job = 16, 10
folder = f'../../data/3ncut_two_zeropi/truc1=300_truc2=1000_pick=True/'
hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()[:truc_full]
eket_tot = ssp.csr_matrix(np.load(folder+ 'eket_tot.npy'))[:truc_full]
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()[:truc_full]
n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
dim_0 = len(hspace_0)
dim_1 = len(hspace_1)
n_theta0_dress = ut.truncate_2(n_theta0_dress, np.arange(truc_full))
n_theta1_dress = ut.truncate_2(n_theta1_dress, np.arange(truc_full))

### cz part
cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
x0_vec = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()[[0],:]
# [[-4],:]   [0::2,:] [[0,3,4, 16],:]

W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]
drive_term = n_theta1_dress
hspace_select = [
    ### state_all_1000
    ### charge_pick
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
# '1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
# '8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
# '0-18', '2-9', '1-12', '0-20', '0-21',
# '18-0', '9-2', '12-1', '5-5', '0-24' ,

# '4-8', '1-13', '20-0', '8-4', '1-16',
# '22-0', '2-12', '13-1', '24-0', '0-25' ,
# '15-1', '0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0' ,
# '13-2', '26-0', '1-18', '15-2', '0-28', '5-9', '4-12', '28-0', '0-30', '18-1' ,
# '12-4', '9-5', '1-20', '1-21', '0-33',
# '8-8', '0-34', '0-35', '1-24', '2-18' ,
# '0-36', '20-1', '4-13', '22-1', '4-16',
# '30-0', '13-4', '5-12', '15-4', '24-1' ,
]

### common part
logi_state = ['0-0', '0-2', '2-0', '2-2']
index_select = [hspace_full.index(i) for i in hspace_select]
len_select = len(hspace_select)
H0_full = qt.Qobj(np.diag(eval_tot))
H0_select = ut.truncate_2( H0_full, index_select)
drive_select = ut.truncate_2(drive_term, index_select)
eket_tot = eket_tot[index_select]
logi_idx_select = [hspace_select.index(i) for i in logi_state]
H_drive_select = [ H0_select,   [drive_select, ut.drive_gauss_A] ]
print('truc_full=', truc_full )
print('num_cpus=', num_cpus, ', n_job=', n_job)
print('params =')
for para in x0_vec:
    print(para.tolist(), ',')
print(f'\nhspace_select (len={len_select}) = [')
for i in range(0, len(hspace_select), 10):
    print(", ".join(f"'{x}'" for x in hspace_select[i:i + 10]), ',')
print(']')
states_all_index = [hspace_full.index(i) for i in hspace_select]
data = states_all_index
print(f'\nhspace_select_index = [')
for i in range(0, len(data), 10):  # Step size of 10
    print(", ".join(f"{x}" for x in data[i:i + 10]), ',')
print(']')


truc_full= 500
num_cpus= 16 , n_job= 10
params =
[20.032421, 0.045974, 0.029778] ,

hspace_select (len=20) = [
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
]

hspace_select_index = [
0, 1, 2, 3, 4, 5, 6, 7, 8, 9 ,
10, 11, 12, 13, 14, 15, 16, 17, 18, 19 ,
]


In [ ]:
t1 = 170    
gamma = 1 / 1e3 / t1 # calculate decay rate given T1, unit in micro-second

folder = f'../../data/3ncut_two_zeropi/truc1=500/'
n_theta0 = np.load(folder+'n_theta0.npy')
n_theta1 = np.load(folder+'n_theta1.npy')
Gamma_decay_0 = gamma / (np.abs(n_theta0[4,8])**2)
Gamma_decay_1 = gamma / (np.abs(n_theta1[4,8])**2)
gamma_decay_0 = Gamma_decay_0 * np.abs(n_theta0) ** 2
gamma_decay_1 = Gamma_decay_1 * np.abs(n_theta1) ** 2

folder = f'../../data/3ncut_two_zeropi/truc1=500/'
gamma_q0 = pd.read_csv(folder+ 'data_gamma_qubit0.txt')
gamma_q1 = pd.read_csv(folder+ 'data_gamma_qubit1.txt')
gamma_dephase_0 = gamma_q0['tphi_02'].to_numpy() *50 /t1
gamma_dephase_1 = gamma_q1['tphi_02'].to_numpy() *50 /t1


In [ ]:
jump_t1 = []
jump_tphi = []
for i in range(1, len(hspace_select)): # take |1,2><3,4| as an example: '1-2'--'3,4'
    si_1, si_2 = map(int, hspace_select[i].split('-')) # extract '1' and '2' for state '1-2'
    for j in range(i):
        sj_1, sj_2 = map(int, hspace_select[j].split('-')) # extract '3' and '4' for state '3-4'
        # t_1
        jop_1 = np.sqrt(gamma_decay_0[si_1, sj_1])* qt.basis(dim_0, si_1) * qt.basis(dim_0, sj_1).dag() # |1><3| for qubit 1
        jop_2 = np.sqrt(gamma_decay_1[si_2, sj_2])* qt.basis(dim_1, si_2) * qt.basis(dim_1, sj_2).dag() # |2><4| for qubit 2
        jop = qt.tensor(jop_1, jop_2) # get tensored jump op
        jump_t1.append(qt.Qobj( eket_tot @ jop.data @ eket_tot.conj().T ))
    # t_phi
    proj_0 = np.sqrt(2*gamma_dephase_0[si_1] )* qt.basis(dim_0, si_1).proj()
    proj_1 = np.sqrt(2*gamma_dephase_1[si_2] )* qt.basis(dim_1, si_2).proj()
    proj = qt.tensor(proj_0, proj_1)
    jump_tphi.append(qt.Qobj( eket_tot @ proj.data @ eket_tot.conj().T))        

In [4]:
np.shape(jump_t1), np.shape(jump_tphi)

((190, 20, 20), (19, 20, 20))

In [3]:
### cz part
# ### ideal fidelity
# c_op_list = [qt.Qobj(np.zeros((len_select, len_select)))]
c_op_list = []
arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select]
f_ideal = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_optimize)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f'\nf_ideal (dim={len(hspace_select)})  = [')
for i in range(0, len(f_ideal), 4):
    print(', '.join(map(str, np.round(f_ideal[i:i+4], 8).tolist())), ',')
print(']')


f_ideal (dim=100)  = [
-0.68353533 ,
]


In [ ]:
#################################################################
### Noisey fidelity
t1_tphi_other = 170 # μs

print("t1_tphi_other = ", t1_tphi_other)
folder = f'../../data/3ncut_two_zeropi/truc1=500/'
gamma_q0 = pd.read_csv(folder+ 'data_gamma_qubit0.txt')
gamma_q1 = pd.read_csv(folder+ 'data_gamma_qubit1.txt')
gamma_decay_48_q0 = gamma_q0['t1_50us_48'].to_numpy() *50 /t1_tphi_other
gamma_decay_48_q1 = gamma_q1['t1_50us_48'].to_numpy() *50 /t1_tphi_other
gamma_dephase_02_q0 = gamma_q0['tphi_02'].to_numpy() *50 /t1_tphi_other
gamma_dephase_02_q1 = gamma_q1['tphi_02'].to_numpy() *50 /t1_tphi_other

qubit_a = True
arg_a = [dim_0, dim_1, gamma_decay_48_q0, gamma_dephase_02_q0, eket_tot, qubit_a] # old gamma
jump_op_a = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *arg_a) for state in range(1,dim_0))

qubit_a = False
arg_b = [dim_0, dim_1, gamma_decay_48_q1, gamma_dephase_02_q1, eket_tot, qubit_a] # old gamma
jump_op_b = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *arg_b) for state in range(1,dim_1))

jump_t1_list = np.array(jump_op_a)[:,0].tolist() + np.array(jump_op_b)[:,0].tolist()
jump_tphi_list = np.array(jump_op_a)[:,1].tolist() + np.array(jump_op_b)[:,1].tolist()

jump_t1_list = [qt.Qobj(matrix) for matrix in jump_t1_list]
jump_tphi_list = [qt.Qobj(matrix) for matrix in jump_tphi_list]


t1_tphi_other =  170


In [7]:
np.shape(jump_op_a)

(153, 2, 100, 100)

In [ ]:
def get_jump_op_charge_pick(state, *args):

    dim_0, dim_1, gamma_decay, gamma_dephase, eket_tot, qubit_a = args
    if qubit_a:
        # t_1
        ladder_0i = qt.basis(dim_0,0) * qt.basis(dim_0, state).dag()
        a_0i_I = qt.tensor(ladder_0i, qt.qeye(dim_1))
        jump_t1 = qt.Qobj( eket_tot @ ( np.sqrt(gamma_decay[state])* a_0i_I ).data @ eket_tot.conj().T )
        # t_phi
        proj_ii = qt.basis(dim_0, state).proj()
        a_ii_I = qt.tensor(proj_ii, qt.qeye(dim_1))
        jump_tphi = qt.Qobj( eket_tot @ ( np.sqrt(2*gamma_dephase[state] )* a_ii_I  ).data @ eket_tot.conj().T)
    else: # qubit_b
        # t_1
        ladder_0j = qt.basis(dim_1,0) * qt.basis(dim_1, state).dag()
        a_I_0i = qt.tensor(qt.qeye(dim_0), ladder_0j)
        jump_t1 = qt.Qobj( eket_tot @ ( np.sqrt(gamma_decay[state])* a_I_0i ).data @ eket_tot.conj().T )
        # t_phi
        proj_ii = qt.basis(dim_1, state).proj()
        a_I_ii = qt.tensor(qt.qeye(dim_0), proj_ii)
        jump_tphi = qt.Qobj( eket_tot @ ( np.sqrt(2*gamma_dephase[state] )* a_I_ii  ).data @ eket_tot.conj().T)
    return [jump_t1, jump_tphi]

In [5]:

c_op_list = jump_t1_list + jump_tphi_list
arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_optimize)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f'\nf_noise (dim={len(hspace_select)})  = [')
for i in range(0, len(f_noise), 4):
    print(', '.join(map(str, np.round(f_noise[i:i+4], 8).tolist())), ',')
print(']')

KeyboardInterrupt: 